# Resume Parser CV TIẾNG VIỆT — pipeline chuẩn (Colab)

**Quy trình:** sinh FILE PDF thật **A4 dọc** (render HTML/CSS bằng Chromium, 4 mẫu: teal bán hàng, navy sinh viên, English SE, TopCV tư vấn) → **trích text từ PDF (PyMuPDF)** → dò nhãn → train NER (xlm-roberta-base) → test.

**Chạy:** *Runtime → Change runtime type → GPU* rồi *Run all*.

> Script Playwright được **ghi ra file rồi chạy bằng `!python`** (chạy inline sẽ lỗi *Sync API inside asyncio loop*).

Nhãn: `Name, Designation, Email Address, Location, College Name, Degree, Graduation Year, Companies worked at, Skills, Years of Experience`.

## 0. Cài đặt (thư viện + Chromium) & GPU

In [ ]:
!pip -q install playwright pymupdf spacy spacy-transformers scikit-learn
!playwright install chromium
!playwright install-deps chromium


In [ ]:
!nvidia-smi


## Bước 1 — Ghi & chạy bộ sinh PDF (A4 dọc)
Xuất `cv_pdfs/*.pdf` + `cv_pdfs/gt/*.json`. Đổi `N_CV` trong file nếu muốn nhiều hơn.

In [ ]:
%%writefile gen_cv_pdfs.py
# -*- coding: utf-8 -*-
"""
BƯỚC 1 — Sinh NHIỀU FILE PDF CV THẬT (khổ A4 DỌC), render bằng Chrome (Playwright)
từ 4 template HTML/CSS mô phỏng SÁT các mẫu người dùng gửi:
  teal   -> mẫu #1 (bán hàng, sidebar teal, ảnh, kỹ năng có sao)
  navy   -> mẫu #2 (sinh viên part-time, sidebar navy, Hoạt động)
  en     -> mẫu #4 (Software Engineer, tiếng Anh, 1 cột)
  topcv  -> mẫu #5 (tư vấn/CSKH kiểu TopCV, serif, kỹ năng có mô tả)

Mỗi PDF kèm ground-truth cv_pdfs/gt/*.json để BƯỚC 2 dò nhãn trên text trích từ PDF.

Chạy: python gen_cv_pdfs.py
Cần: pip install playwright  (+ Chrome sẵn có; hoặc trên Colab: playwright install chromium)
"""
import os, json, random
from playwright.sync_api import sync_playwright

random.seed(42)
OUT_DIR = os.path.dirname(os.path.abspath(__file__))
PDF_DIR = os.path.join(OUT_DIR, "cv_pdfs")
GT_DIR  = os.path.join(PDF_DIR, "gt")
os.makedirs(GT_DIR, exist_ok=True)
N_CV = 300
WEIGHTS = [("teal",34),("topcv",30),("navy",22),("en",14)]  # tỉ lệ mỗi template

# ---------------------------------- ẢNH AVATAR -----------------------------
AV_SQ=("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='200' height='230'>"
 "<rect width='200' height='230' fill='%232f5058'/><circle cx='100' cy='85' r='48' fill='%23c9d6d9'/>"
 "<path d='M40 210 Q100 130 160 210 Z' fill='%23c9d6d9'/></svg>")
AV_CI=("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='150' height='150'>"
 "<circle cx='75' cy='75' r='75' fill='%23dfe4ee'/><circle cx='75' cy='60' r='30' fill='%239aa6c0'/>"
 "<path d='M25 140 Q75 90 125 140 Z' fill='%239aa6c0'/></svg>")

# ---------------------------------- KHO DỮ LIỆU ----------------------------
HO=["Nguyễn","Trần","Lê","Phạm","Hoàng","Huỳnh","Phan","Vũ","Võ","Đặng","Bùi","Đỗ","Ngô","Dương","Lý","Mai"]
TEN_NAM=["Văn An","Hoàng Nam","Minh Quân","Đức Anh","Quốc Bảo","Xuân Thưởng","Nhật Minh","Thành Đạt","Gia Huy","Tuấn Kiệt","Đăng Khoa","Trọng Nghĩa"]
TEN_NU=["Trúc Quỳnh My","Ngọc Linh","Minh Trang","Thu Hà","Phương Anh","Thảo Nguyên","Khánh Vy","Bảo Ngọc","Mai Chi","Hồng Nhung","Yến Nhi","Kim Ngân"]
CITIES=["Hà Nội","TP.HCM","Đà Nẵng","Hải Phòng","Cần Thơ","Nha Trang","Huế"]
DIST={"Hà Nội":["Cầu Giấy","Hoàng Mai","Đống Đa","Thanh Xuân","Hà Đông"],"TP.HCM":["Quận 1","Quận 3","Quận 10","Bình Thạnh","Tân Bình"],
      "Đà Nẵng":["Hải Châu","Sơn Trà","Thanh Khê"],"Hải Phòng":["Lê Chân","Ngô Quyền"],"Cần Thơ":["Ninh Kiều","Cái Răng"],
      "Nha Trang":["Lộc Thọ","Vĩnh Hải"],"Huế":["Phú Hội","Vĩnh Ninh"]}
STREET=["Nguyễn Chí Thanh","Trần Hưng Đạo","Lê Lợi","Nguyễn Trãi","Hoàng Diệu","Nguyễn Văn Cừ","Lê Duẩn"]
UNIS=["Trường Đại học Bách khoa Đà Nẵng","Trường Đại học Bách khoa Hà Nội","Trường Đại học Xây dựng Hà Nội",
      "Trường Đại học Kinh tế Quốc dân","Trường Đại học Ngoại thương","Trường Đại học Kinh tế TP.HCM",
      "Trường Đại học Công nghệ Thông tin - ĐHQG TP.HCM","Trường Đại học FPT","Trường Đại học Thương mại"]
LOAI=["Xuất sắc","Giỏi","Khá"]

SALES_TITLES=["Nhân viên bán hàng","Nhân viên kinh doanh","Chuyên viên tư vấn bán hàng","Nhân viên tư vấn"]
SALES_EXP=["Nhân viên kinh doanh","Marketing Manager","Chuyên viên bán hàng","Trưởng nhóm bán hàng"]
SALES_SKILLS=["Quản lý dự án.","Giao tiếp tốt","Nắm bắt kiến thức sản phẩm nhanh","Kỹ năng thuyết phục","Quản lý thời gian","Kỹ năng đàm phán"]
SALES_CO=["Công ty CP Thương mại ABC","Công ty TNHH Phân phối Hòa Bình","Công ty CP Bán lẻ Thế Giới Số","Công ty TNHH Thương mại Minh Phát"]
SALES_BULLET=["Phụ trách việc tìm kiếm và khai thác thị trường mới, xây dựng danh sách khách hàng tiềm năng: Tại công ty ABC, tôi đã chịu trách nhiệm tìm kiếm và khai thác các thị trường mới, từ đó tạo ra danh sách khách hàng tiềm năng.",
              "Tôi đã nghiên cứu và đánh giá các xu hướng thị trường để xác định các cơ hội kinh doanh mới. Tôi đã áp dụng các kỹ thuật tiếp thị và xây dựng mạng lưới khách hàng để tăng doanh số bán hàng."]
STU_SKILLS=["Sử dụng máy tính và các công cụ bán hàng cơ bản","Giao tiếp cơ bản, thái độ thân thiện","Khả năng học nhanh và tiếp thu công việc mới","Quản lý thời gian học tập và làm việc"]
STU_ACT=[("Hỗ trợ bán hàng","Chương trình gây quỹ của lớp",["Chuẩn bị hàng hóa, sắp xếp quầy bán và hỗ trợ tư vấn sản phẩm cho người mua.","Tham gia thu ngân, ghi nhận đơn hàng và kiểm soát số lượng bán ra."]),
         ("Phụ bán hàng","Cửa hàng gia đình",["Hỗ trợ tiếp đón khách, giới thiệu sản phẩm và giải đáp các câu hỏi cơ bản.","Rèn luyện kỹ năng giao tiếp, thái độ phục vụ và xử lý tình huống với khách hàng."]),
         ("Cộng tác viên truyền thông","Câu lạc bộ Sự kiện của trường",["Tham gia tổ chức sự kiện và truyền thông trên fanpage của câu lạc bộ.","Phối hợp cùng các thành viên để hoàn thành công việc được giao."])]
IT_TITLES_EN=["Senior Software Engineer","Software Engineer","Back-end Developer","Full-stack Developer"]
IT_EXP_EN=["Senior Software Engineer","Back-end Developer","Software Engineer","QA Engineer"]
IT_SKILLS=["Java","Python","JavaScript","ReactJS","Spring Boot","SQL","Git","Docker","C#",".NET Core","NodeJS","PostgreSQL","MongoDb","AngularJS","Selenium"]
IT_CO=["TopDev","Applancer Joint Stock Company","FPT Software","VNG Corporation","Viettel Solutions"]
IT_DEG_EN=["Information Technology","Computer Science","Software Engineering","Information Systems"]
EN_BULLET=["Choose technologies and build backend project structure with Spring, Mongodb, Restful web service.",
           "Research and apply automation test tools; write automation test scripts for the test management system.",
           "Developing back-end (NodeJS or PHP) and working directly with Product Owner and UI/UX Designer.",
           "Participate in the complete software development cycle: requirement analysis, coding, testing, deployment."]
CSKH_TITLES=["Nhân viên tư vấn","Tổng đài viên Chăm sóc khách hàng","Chuyên viên tư vấn"]
CSKH_EXP=["Tổng Đài Viên Chăm Sóc Khách Hàng","Chuyên Viên Tư Vấn Giải Pháp Phần Mềm","Nhân Viên Chăm Sóc Khách Hàng"]
CSKH_CO=["SVT Finance Innovation Co.","SVT Investment & Development Co., Ltd","Công ty CP Giáo dục Everest","Ngân hàng TMCP Kỹ Thương"]
CSKH_BULLET=["Tiếp nhận và giải đáp thắc mắc của khách hàng liên quan đến sản phẩm, dịch vụ qua điện thoại, Zalo và email.",
             "Theo dõi mức độ hài lòng của khách hàng, ghi nhận các phản hồi và đề xuất cải thiện trải nghiệm dịch vụ.",
             "Cập nhật và quản lý thông tin khách hàng trên hệ thống CRM, đảm bảo dữ liệu đầy đủ, chính xác và dễ truy xuất.",
             "Phân tích nhu cầu và quy trình quản lý nhân sự của khách hàng, đề xuất giải pháp phần mềm phù hợp.",
             "Tư vấn giải pháp và trình bày demo sản phẩm cho khách hàng doanh nghiệp."]
CSKH_SKILLS=[("Kỹ năng giao tiếp","Thành thạo trong việc lắng nghe, truyền đạt thông tin rõ ràng và thuyết phục"),
             ("Phân tích và giải quyết vấn đề","Có khả năng phân tích nhu cầu khách hàng, đánh giá tình huống và đề xuất các giải pháp"),
             ("Thuyết phục và đàm phán","Thành thạo trong việc thuyết phục khách hàng và đàm phán để đạt được thỏa thuận đôi bên cùng có lợi"),
             ("Sử dụng CRM","Sử dụng thành thạo hệ thống CRM để quản lý và tra cứu thông tin khách hàng")]

def nm(): return f"{random.choice(HO)} {random.choice(TEN_NAM if random.random()<.5 else TEN_NU)}"
def phone(): return "0"+"".join(str(random.randint(0,9)) for _ in range(9))
def email(n):
    tbl=str.maketrans("àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ",
                      "aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd")
    return "".join(w.translate(tbl) for w in n.lower().split())+random.choice(["",str(random.randint(1,99))])+"@"+random.choice(["gmail.com","gmail.com","outlook.com"])
def city2(): c=random.choice(CITIES); return random.choice(DIST[c]), c
def dr():
    y1=random.randint(2016,2022); y2=y1+random.choice([1,1,2]); return "%02d/%d - %02d/%d"%(random.randint(1,12),y1,random.randint(1,12),y2)
def dedup(x): return list(dict.fromkeys(x))

# --------------------------------- TEMPLATE HTML ---------------------------
def teal_html(d):
    sk="".join(f"<div class='sk'>{s}</div>" for s in d["skills"])
    st="".join(f"<div class='skstar'><span class='dot'></span>{s}<span class='star'>{r} ★</span></div>" for s,r in d["skills_star"])
    ex=""
    for e in d["experiences"]:
        bl="".join(f"<li>{b}</li>" for b in e["bullets"])
        ex+=f"<div class='exp'><div class='exprow'><span class='co'>{e['company']}</span><span class='dt'>{e['dates']}</span></div><div class='role'>{e['title']}</div><ul>{bl}</ul></div>"
    return f"""<!doctype html><html><head><meta charset='utf-8'>
<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@500;600;700&family=Roboto:ital,wght@0,300;0,400;0,700;1,400&display=swap" rel="stylesheet">
<style>@page{{size:A4;margin:0}}*{{margin:0;padding:0;box-sizing:border-box}}
body{{font-family:'Roboto',sans-serif;color:#2b2b2b;font-size:11.5px;line-height:1.5;width:794px}}
.wrap{{display:flex;min-height:1123px}}
.side{{width:34%;background:#41707a;color:#12333a;padding:0 22px 30px}}
.photo{{width:170px;height:190px;margin:26px auto 20px;display:block;border:2px solid rgba(255,255,255,.3)}}
.side h2{{font-family:'Oswald';font-weight:600;font-size:20px;color:#123;margin:18px 0 8px}}
.lbl{{font-weight:700;margin-top:9px;color:#0f2c31}}.val{{color:#20474e}}
.sk{{margin-top:5px;color:#173e44}}
.skstar{{display:flex;align-items:center;color:#173e44;margin-top:6px;font-size:11px}}
.skstar .dot{{width:5px;height:5px;border-radius:50%;background:#123;display:inline-block;margin-right:7px}}
.skstar .star{{margin-left:auto;color:#0f2c31;font-size:10px;opacity:.75}}
.main{{width:66%;padding:40px 40px 30px}}
.name{{font-family:'Oswald';font-weight:700;font-size:40px;line-height:1.02;color:#2f5b63;letter-spacing:1px;text-transform:uppercase}}
.subt{{letter-spacing:5px;color:#7d7d7d;font-size:12px;margin:8px 0 26px;text-transform:uppercase}}
.h{{font-family:'Oswald';font-weight:600;font-size:19px;margin:0 0 8px}}
.bar{{background:#9db9bd;color:#173e44;font-family:'Oswald';font-weight:600;font-size:17px;padding:7px 14px;margin:24px 0 14px}}
.obj{{color:#555;margin-bottom:6px}}
.exprow{{display:flex;justify-content:space-between;align-items:baseline}}.co{{font-weight:700}}.dt{{color:#888;font-size:11px}}
.role{{font-style:italic;color:#666;margin:1px 0 5px}}.exp ul{{margin:0 0 14px 16px}}.exp li{{margin-bottom:6px;color:#555}}
</style></head><body><div class='wrap'>
<div class='side'><img class='photo' src="{AV_SQ}"/>
<h2>Liên lạc</h2>
<div class='lbl'>Điện thoại</div><div class='val'>{d['phone']}</div>
<div class='lbl'>Email</div><div class='val'>{d['email']}</div>
<div class='lbl'>Ngày sinh</div><div class='val'>{d['dob']}</div>
<div class='lbl'>Địa chỉ</div><div class='val'>{d['address']}</div>
<h2>Kỹ năng</h2>{sk}<h2>Kỹ năng</h2>{st}</div>
<div class='main'><div class='name'>{d['name']}</div><div class='subt'>{d['title']}</div>
<div class='h'>Mục tiêu nghề nghiệp</div><div class='obj'>{d['objective']}</div>
<div class='bar'>Kinh nghiệm làm việc</div>{ex}</div></div></body></html>"""

def navy_html(d):
    sk="".join(f"<li>{s}</li>" for s in d["skills"])
    ac=""
    for a in d["activities"]:
        bl="".join(f"<li>{b}</li>" for b in a["bullets"])
        ac+=f"<div class='act'><div class='yr'>{a['year']}</div><div class='ab'><div class='ar'>{a['role']}</div><div class='ao'>{a['org']}</div><ul>{bl}</ul></div></div>"
    return f"""<!doctype html><html><head><meta charset='utf-8'>
<link href="https://fonts.googleapis.com/css2?family=Montserrat:wght@600;700;800&family=Roboto:wght@300;400;500&display=swap" rel="stylesheet">
<style>@page{{size:A4;margin:0}}*{{margin:0;padding:0;box-sizing:border-box}}
body{{font-family:'Roboto',sans-serif;font-size:11px;color:#333;line-height:1.5;width:794px}}
.wrap{{display:flex;min-height:1123px}}
.side{{width:33%;background:#1f3a6e;color:#eef2fa;padding:0 20px 30px}}
.photo{{width:120px;height:120px;border-radius:50%;display:block;margin:28px auto 18px;border:3px solid #fff}}
.sh{{background:#3a5aa0;font-family:'Montserrat';font-weight:700;font-size:13px;padding:5px 12px;margin:18px -20px 10px;color:#fff}}
.lbl{{font-weight:700;margin-top:8px;color:#fff}}.val{{color:#c9d4ea;font-size:10.5px}}
.side ul{{margin-left:16px}}.side li{{margin-top:5px;color:#dfe6f5}}
.edu b{{color:#fff}}.edu div{{color:#c9d4ea}}
.main{{width:67%;padding:38px 40px}}
.name{{font-family:'Montserrat';font-weight:800;font-size:33px;line-height:1.03;color:#1f3a6e;text-transform:uppercase}}
.subt{{color:#666;font-size:13px;margin:6px 0 24px}}
.h{{font-family:'Montserrat';font-weight:700;color:#1f3a6e;font-size:17px;margin-bottom:8px}}
.obj{{color:#555;margin-bottom:26px}}
.act{{display:flex;margin-bottom:16px}}.yr{{width:64px;color:#1f3a6e;font-weight:700;font-size:11px;flex-shrink:0}}
.ar{{font-weight:700}}.ao{{font-style:italic;color:#777;font-size:10.5px}}
.ab ul{{margin:5px 0 0 16px}}.ab li{{margin-bottom:4px;color:#555}}
</style></head><body><div class='wrap'>
<div class='side'><img class='photo' src="{AV_CI}"/>
<div class='sh'>Contact</div>
<div class='lbl'>Số điện thoại</div><div class='val'>{d['phone']}</div>
<div class='lbl'>Email</div><div class='val'>{d['email']}</div>
<div class='lbl'>Địa chỉ</div><div class='val'>{d['location']}</div>
<div class='sh'>Học vấn</div><div class='edu'><b>{d['college']}</b><div>Ngành: {d['degree']}</div><div>{d['edu']}</div></div>
<div class='sh'>Kỹ năng</div><ul>{sk}</ul>
<div class='sh'>Ngôn ngữ</div><div class='val'>Tiếng Anh</div></div>
<div class='main'><div class='name'>{d['name']}</div><div class='subt'>{d['title']}</div>
<div class='h'>Mục tiêu học tập – nghề nghiệp</div><div class='obj'>{d['objective']}</div>
<div class='h'>Hoạt động - Kinh nghiệm liên quan</div>{ac}</div></div></body></html>"""

def en_html(d):
    ex=""
    for e in d["experiences"]:
        bl="".join(f"<li>{b}</li>" for b in e["bullets"])
        ex+=f"<div class='exprow'><b>{e['company']}</b><span class='dt'>{e['dates']}</span></div><div class='role'>{e['title']}</div><div class='rsp'>Responsibilities:</div><ul>{bl}</ul><div class='tech'>Technologies: {', '.join(e['tech'])}</div>"
    return f"""<!doctype html><html><head><meta charset='utf-8'>
<style>@page{{size:A4;margin:0}}*{{margin:0;padding:0;box-sizing:border-box}}
body{{font-family:Arial,Helvetica,sans-serif;font-size:11px;color:#111;line-height:1.45;padding:42px 54px;width:794px}}
.name{{text-align:center;font-weight:700;font-size:22px;letter-spacing:1px}}
.subt{{text-align:center;font-weight:700;font-size:12px;margin-top:2px}}
.contact{{text-align:center;font-size:10px;color:#333;margin:8px 0 4px}}
.contact2{{text-align:center;font-size:10px;color:#333;margin-bottom:16px}}
.h{{font-weight:700;font-size:13px;border-bottom:1.5px solid #111;padding-bottom:3px;margin:16px 0 8px}}
.exprow{{display:flex;justify-content:space-between;margin-top:10px}}.dt{{font-weight:700}}
.role{{font-weight:700;font-style:italic;margin:1px 0}}.rsp{{margin-top:4px}}
ul{{margin:3px 0 3px 20px}}li{{margin-bottom:2px}}.tech{{margin:4px 0 2px}}
</style></head><body>
<div class='name'>{d['name']}</div><div class='subt'>{d['title']}</div>
<div class='contact'>{d['phone']} - {d['email']} - {d['location']}</div>
<div class='contact2'>{d['dob']} - github.com/{d['slug']} - linkedin.com/in/{d['slug']}</div>
<div class='h'>SUMMARY</div><div>{d['summary']}</div>
<div class='h'>WORK EXPERIENCE</div>{ex}
<div class='h'>EDUCATION</div>
<div class='exprow'><b>{d['college']}</b><span class='dt'>{d['grad']}</span></div><div class='role'>{d['degree']}</div>
</body></html>"""

def topcv_html(d):
    ex=""
    for e in d["experiences"]:
        bl="".join(f"<li>{b}</li>" for b in e["bullets"])
        ex+=f"<div class='exprow'><b>{e['company']}</b><span class='dt'>{e['dates']}</span></div><div class='role'>{e['title']}</div><ul>{bl}</ul>"
    sk="".join(f"<div class='skrow'><div class='skn'>{n}</div><div class='skd'>{v}</div></div>" for n,v in d["skills"])
    return f"""<!doctype html><html><head><meta charset='utf-8'>
<link href="https://fonts.googleapis.com/css2?family=Roboto+Slab:wght@400;700&family=PT+Serif:ital,wght@0,400;0,700;1,400&display=swap" rel="stylesheet">
<style>@page{{size:A4;margin:0}}*{{margin:0;padding:0;box-sizing:border-box}}
body{{font-family:'PT Serif',serif;font-size:11.5px;color:#1a1a1a;line-height:1.5;padding:34px 48px;width:794px}}
.name{{font-family:'Roboto Slab';text-align:center;font-weight:700;font-size:26px}}
.subt{{text-align:center;font-style:italic;font-size:13px;margin-top:2px}}
.contact{{text-align:center;font-size:11px;margin:12px 0 6px;color:#222}}
.h{{font-family:'Roboto Slab';font-weight:700;font-size:15px;border-bottom:1px solid #111;padding-bottom:3px;margin:20px 0 9px}}
.exprow{{display:flex;justify-content:space-between;margin-top:9px}}.dt{{color:#333}}
.role{{font-weight:700;margin:2px 0 3px}}ul{{margin:0 0 4px 18px}}li{{margin-bottom:4px;color:#222}}
.edurow{{display:flex;justify-content:space-between}}.edudeg{{font-weight:700;margin-top:3px}}
.skrow{{display:flex;border-bottom:1px solid #eee;padding:6px 0}}.skn{{width:34%;font-weight:700}}.skd{{width:66%;color:#333}}
</style></head><body>
<div class='name'>{d['name']}</div><div class='subt'>{d['title']}</div>
<div class='contact'>&#9742; {d['phone']} &nbsp;&nbsp; &#9993; {d['email']} &nbsp;&nbsp; &#128279; be.net/tencuaban &nbsp;&nbsp; &#128205; {d['location']}</div>
<div class='h'>MỤC TIÊU NGHỀ NGHIỆP</div><div>{d['objective']}</div>
<div class='h'>HỌC VẤN</div><div class='edurow'><b>{d['college']}</b><span class='dt'>{d['edu']}</span></div>
<div class='edudeg'>{d['degree']}</div><div>Tốt nghiệp loại {d['loai']}</div>
<div class='h'>KINH NGHIỆM LÀM VIỆC</div>{ex}
<div class='h'>KỸ NĂNG</div>{sk}
<div class='h'>HOẠT ĐỘNG</div><div class='exprow'><b>{d['activity']}</b><span class='dt'>{d['act_dates']}</span></div>
</body></html>"""

# ------------------------------ SINH DỮ LIỆU + GT --------------------------
def build_teal():
    n=nm(); t=random.choice(SALES_TITLES); dist,city=city2()
    loc=f"{dist}, {city}"; addr=f"{random.choice(STREET)}, Phường {random.randint(1,15)}, {loc}"
    sks=random.sample(SALES_SKILLS,random.randint(5,6))
    star=[("Microsoft Word",random.choice(["4","4.5","5"])),("Microsoft Excel",random.choice(["4","4.5","5"]))]
    exps=[{"company":c,"title":random.choice(SALES_EXP),"dates":dr(),"bullets":SALES_BULLET} for c in random.sample(SALES_CO,random.choice([1,2]))]
    obj=(f"Tôi là một {t.lower()} chuyên nghiệp, đam mê trong việc xây dựng mối quan hệ với khách hàng và đạt "
         f"được mục tiêu doanh số. Mục tiêu của tôi là phát triển sự nghiệp trong lĩnh vực bán hàng, áp dụng kỹ "
         f"năng giao tiếp mạnh mẽ và khả năng thuyết phục để tạo ra giá trị cho khách hàng và đóng góp vào sự "
         f"thành công của tổ chức.")
    d={"name":n,"title":t,"phone":phone(),"email":email(n),"dob":"%02d/%02d/%d"%(random.randint(1,28),random.randint(1,12),random.randint(1996,2002)),
       "address":addr,"skills":sks,"skills_star":star,"objective":obj,"experiences":exps}
    gt={"Name":[n],"Designation":dedup([t]+[e["title"] for e in exps]),"Email Address":[d["email"]],
        "Location":[loc],"Skills":dedup([s.rstrip(".") for s in sks]+[s for s,_ in star]),
        "Companies worked at":dedup([e["company"] for e in exps])}
    return "teal",d,gt

def build_navy():
    n=nm(); t=random.choice(SALES_TITLES)+random.choice([""," part-time"]); dist,city=city2()
    loc=f"{dist}, {city}"; s1=random.randint(2021,2023); grad=str(s1+4)
    col=random.choice(UNIS); deg=random.choice(["Công nghệ thông tin","Quản trị kinh doanh","Marketing","Kinh tế"])
    sks=random.sample(STU_SKILLS,min(4,len(STU_SKILLS)))
    acts=random.sample(STU_ACT,2)
    years=["2025","2019 - Nay","2022","2021 - 2023"]
    activities=[{"year":years[i],"role":a[0],"org":a[1],"bullets":a[2]} for i,a in enumerate(acts)]
    obj=(f"Sinh viên năm cuối mong muốn ứng tuyển vị trí {t} để rèn luyện kỹ năng giao tiếp với khách hàng, "
         f"tác phong làm việc chuyên nghiệp và khả năng quản lý thời gian, đồng thời lấy kinh nghiệm thực tế "
         f"trong môi trường dịch vụ.")
    d={"name":n,"title":t,"phone":phone(),"email":email(n),"location":loc,"college":col,"degree":deg,
       "edu":f"{s1} – {grad}","skills":sks,"objective":obj,"activities":activities}
    gt={"Name":[n],"Designation":dedup([t]+[a["role"] for a in activities]),"Email Address":[d["email"]],
        "Location":[loc],"College Name":[col],"Degree":[deg],"Graduation Year":[grad],"Skills":dedup(sks)}
    return "navy",d,gt

def build_en():
    n=nm(); t=random.choice(IT_TITLES_EN); dist,city=city2(); loc=f"{dist}, {city}"
    ny=random.randint(3,8); grad=str(random.randint(2013,2020))
    col=random.choice(UNIS); deg=random.choice(IT_DEG_EN)
    exps=[]
    for c in random.sample(IT_CO,2):
        exps.append({"company":c,"title":random.choice(IT_EXP_EN),"dates":dr().replace(" - ","-"),
                     "tech":random.sample(IT_SKILLS,random.randint(3,5)),"bullets":random.sample(EN_BULLET,2)})
    summ=(f"I have {ny} years of work experience in Software Development. I have experience and strong at "
          f"Software and Web Application using Java. I am able to apply automation test frameworks using Java.")
    em=email(n)
    d={"name":n,"title":t,"phone":phone(),"email":em,"location":loc,"slug":em.split("@")[0],
       "dob":"%02d/%02d/%d"%(random.randint(1,28),random.randint(1,12),random.randint(1988,1998)),
       "summary":summ,"experiences":exps,"college":col,"degree":deg,"grad":grad}
    tech=dedup(sum([e["tech"] for e in exps],[]))
    gt={"Name":[n],"Designation":dedup([t]+[e["title"] for e in exps]),"Email Address":[d["email"]],
        "Location":[loc],"College Name":[col],"Degree":[deg],"Graduation Year":[grad],
        "Companies worked at":dedup([e["company"] for e in exps]),"Skills":tech,"Years of Experience":[f"{ny} years"]}
    return "en",d,gt

def build_topcv():
    n=nm(); t=random.choice(CSKH_TITLES); dist,city=city2(); loc=f"{dist}, {city}"
    s1=random.randint(2014,2019); grad=str(s1+4); col=random.choice(UNIS+["ĐH TopCV"])
    deg=random.choice(["Quản trị kinh doanh","Ngôn ngữ Anh","Kinh tế","Quản trị dịch vụ"])
    yoe=random.choice(["2 năm kinh nghiệm","3 năm kinh nghiệm","hơn 3 năm kinh nghiệm","4 năm kinh nghiệm"])
    exps=[]
    cos=random.sample(CSKH_CO,2)
    for i,c in enumerate(cos):
        exps.append({"company":c,"title":random.choice(CSKH_EXP),
                     "dates":(dr().split(" - ")[0]+" - Nay") if i==0 else dr(),
                     "bullets":random.sample(CSKH_BULLET,random.randint(2,3))})
    sks=random.sample(CSKH_SKILLS,3)
    obj=(f"Tôi mong muốn ứng tuyển vào vị trí {t} để tận dụng {yoe} xử lý yêu cầu và phản hồi khách hàng "
         f"chuyên nghiệp, cùng với khả năng giao tiếp tiếng Anh tốt và tinh thần làm việc năng động, linh hoạt. "
         f"Mục tiêu của tôi là không ngừng nâng cao trải nghiệm khách hàng, góp phần xây dựng hình ảnh chuyên "
         f"nghiệp và thân thiện cho doanh nghiệp.")
    d={"name":n,"title":t,"phone":phone(),"email":email(n),"location":loc,"college":col,"degree":deg,
       "edu":f"{s1} - {grad}","loai":random.choice(LOAI),"objective":obj,"experiences":exps,"skills":sks,
       "activity":"Câu lạc bộ Sự kiện, trường "+random.choice(UNIS),"act_dates":dr()}
    gt={"Name":[n],"Designation":dedup([t]+[e["title"] for e in exps]),"Email Address":[d["email"]],
        "Location":[loc],"College Name":[col],"Degree":[deg],"Graduation Year":[grad],
        "Companies worked at":dedup([e["company"] for e in exps]),
        "Skills":dedup([s for s,_ in sks]),"Years of Experience":[yoe]}
    return "topcv",d,gt

BUILDERS={"teal":build_teal,"navy":build_navy,"en":build_en,"topcv":build_topcv}
RENDER={"teal":teal_html,"navy":navy_html,"en":en_html,"topcv":topcv_html}
POOL=[k for k,w in WEIGHTS for _ in range(w)]

# --------------------------------- MAIN ------------------------------------
def main():
    from collections import Counter
    cnt=Counter()
    with sync_playwright() as p:
        try: browser=p.chromium.launch(channel="chrome")
        except Exception: browser=p.chromium.launch()
        page=browser.new_page(viewport={"width":794,"height":1123})
        for i in range(1,N_CV+1):
            layout=random.choice(POOL); cnt[layout]+=1
            _,d,gt=BUILDERS[layout]()
            page.set_content(RENDER[layout](d), wait_until="networkidle")
            stem=f"cv_{i:04d}"
            page.pdf(path=os.path.join(PDF_DIR,stem+".pdf"), format="A4", print_background=True)
            json.dump(gt, open(os.path.join(GT_DIR,stem+".json"),"w",encoding="utf-8"), ensure_ascii=False, indent=1)
        browser.close()
    print(f"Đã sinh {N_CV} PDF (A4 dọc) vào {PDF_DIR}")
    print("Theo template:", dict(cnt))

if __name__=="__main__":
    main()


In [ ]:
!python gen_cv_pdfs.py


In [ ]:
# Xem TẤT CẢ CV đã sinh: render trang 1 của mỗi PDF thành ảnh, xếp lưới cuộn được
import glob, base64, os, fitz
from IPython.display import HTML, display

pdfs = sorted(glob.glob('cv_pdfs/*.pdf'))
print('Tổng PDF:', len(pdfs))

DPI   = 60     # tăng lên 100 nếu muốn nét hơn (ảnh nặng hơn, notebook to hơn)
COLS  = 5      # số cột trong lưới
LIMIT = None   # None = hiện tất cả; đặt số (vd 30) nếu muốn xem thử cho nhanh

cards = []
for f in (pdfs if LIMIT is None else pdfs[:LIMIT]):
    pix = fitz.open(f)[0].get_pixmap(dpi=DPI)
    b64 = base64.b64encode(pix.tobytes('png')).decode()
    cards.append(
        "<figure style='margin:0'>"
        f"<img src='data:image/png;base64,{b64}' style='width:100%;border:1px solid #ccc'/>"
        f"<figcaption style='font:11px sans-serif;text-align:center'>{os.path.basename(f)}</figcaption>"
        "</figure>")

display(HTML(
    "<div style='max-height:820px;overflow:auto;display:grid;"
    f"grid-template-columns:repeat({COLS},1fr);gap:10px;padding:6px'>"
    + "".join(cards) + "</div>"))


In [ ]:
# Hiện TOÀN BỘ CV full-size ngay trong output (ảnh nhúng sẵn -> tải notebook về,
# mở ở Colab/Jupyter khác vẫn xem được hết, không cần chạy lại kernel)
import glob, base64, os, fitz
from IPython.display import display, HTML

DPI     = 80      # 72 ~ vừa đọc; 110 ~ nét, file nặng gấp đôi
LIMIT   = None    # None = tất cả; hoặc đặt số CV muốn hiện
MAX_MB  = 80      # ngưỡng an toàn: notebook quá nặng sẽ khó lưu/tải
FORCE   = False   # True = bỏ qua cảnh báo dung lượng

pdfs = sorted(glob.glob('cv_pdfs/*.pdf'))[:LIMIT]
print(f'Số CV sẽ hiện: {len(pdfs)}  |  DPI={DPI}')

def render(path):
    """Trả về danh sách thẻ <img> base64 cho mọi trang của 1 PDF."""
    with fitz.open(path) as pdf:
        return [f"<img src='data:image/png;base64,"
                f"{base64.b64encode(pg.get_pixmap(dpi=DPI).tobytes('png')).decode()}' "
                f"style='width:794px;max-width:100%;border:1px solid #bbb;display:block;"
                f"margin:0 auto 10px'/>" for pg in pdf]

# --- ước lượng dung lượng trước khi vẽ hết ---
probe = sum(len(''.join(render(p))) for p in pdfs[:3]) / max(min(3, len(pdfs)), 1)
est_mb = probe * len(pdfs) / 1024 / 1024
print(f'Ước lượng output: ~{est_mb:.0f} MB')
if est_mb > MAX_MB and not FORCE:
    raise SystemExit(f'DỪNG: vượt {MAX_MB} MB. Hạ DPI (vd 60), đặt LIMIT, hoặc FORCE=True.')

# --- vẽ từng CV, mỗi CV một khối riêng có tiêu đề ---
for i, p in enumerate(pdfs, 1):
    display(HTML(
        f"<div style='margin:18px 0 6px;font:600 13px sans-serif;color:#1f3a6e'>"
        f"[{i}/{len(pdfs)}] {os.path.basename(p)}</div>" + ''.join(render(p))))

print(f'\nXong {len(pdfs)} CV. Lưu notebook (File → Download .ipynb) là giữ được toàn bộ ảnh.')


## Bước 2 — Ghi & chạy: trích text từ PDF → gán nhãn → train_data_vi.json

In [ ]:
%%writefile pdf_to_dataset.py
# -*- coding: utf-8 -*-
"""
BƯỚC 2 — Từ các PDF đã sinh: TRÍCH TEXT bằng PyMuPDF (giống lúc inference),
rồi DÒ LẠI giá trị ground-truth trong text để gán nhãn (start,end,label).

=> dataset huấn luyện phản ánh đúng text thật khi đọc PDF (thứ tự, xuống dòng...).

Xuất: train_data_vi.json  (định dạng notebook gốc: [ [text, {"entities":[[s,e,l],...]}], ... ])
Chạy: python pdf_to_dataset.py
Yêu cầu: pip install pymupdf
"""
import os, re, json, glob, fitz
from collections import Counter

OUT_DIR = os.path.dirname(os.path.abspath(__file__))
PDF_DIR = os.path.join(OUT_DIR, "cv_pdfs")
GT_DIR  = os.path.join(PDF_DIR, "gt")

# nhãn số/dễ trùng -> chỉ lấy lần xuất hiện đầu để giảm nhiễu
FIRST_ONLY = {"Graduation Year"}

_W = r"0-9A-Za-zÀ-ỹ"  # ký tự "chữ" (gồm tiếng Việt) để chặn khớp nửa từ

def spans_of(text, value, first_only=False):
    """Dò 'value' trong text:
    - khoảng trắng linh hoạt (PDF hay ngắt dòng giữa cụm),
    - KHÔNG phân biệt hoa/thường (header CSS in hoa: 'NGUYỄN...' vẫn khớp 'Nguyễn...'),
    - có ranh giới từ (tránh 'Git' khớp nhầm trong 'github')."""
    toks = value.split()
    if not toks:
        return []
    body = r"\s+".join(re.escape(t) for t in toks)
    pat = re.compile(rf"(?<![{_W}]){body}(?![{_W}])", re.IGNORECASE)
    out = []
    for m in pat.finditer(text):
        out.append((m.start(), m.end()))
        if first_only:
            break
    return out

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    return "\n".join(page.get_text() for page in doc)

def main():
    pdfs = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
    assert pdfs, f"Không thấy PDF trong {PDF_DIR}. Chạy gen_cv_pdfs.py trước."
    data = []
    total_vals = matched_vals = 0
    miss = Counter()
    label_cnt = Counter()

    for p in pdfs:
        stem = os.path.splitext(os.path.basename(p))[0]
        gt = json.load(open(os.path.join(GT_DIR, stem + ".json"), encoding="utf-8"))
        text = extract_text(p)

        cand = []  # (start,end,label)
        for label, values in gt.items():
            fo = label in FIRST_ONLY
            for v in values:
                total_vals += 1
                sp = spans_of(text, v, first_only=fo)
                if sp:
                    matched_vals += 1
                else:
                    miss[label] += 1
                for s, e in sp:
                    cand.append((s, e, label))

        # giải quyết chồng lấn: ưu tiên span DÀI hơn
        cand.sort(key=lambda x: -(x[1] - x[0]))
        occupied, ents = set(), []
        for s, e, l in cand:
            if any(i in occupied for i in range(s, e)):
                continue
            occupied.update(range(s, e))
            ents.append([s, e, l]); label_cnt[l] += 1
        ents.sort()
        data.append([text, {"entities": ents}])

    json.dump(data, open(os.path.join(OUT_DIR, "train_data_vi.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)

    print(f"PDF xử lý: {len(pdfs)}  |  CV có nhãn: {len(data)}")
    print(f"Giá trị dò được: {matched_vals}/{total_vals} ({100*matched_vals/max(total_vals,1):.1f}%)")
    print("Nhãn thu được:", dict(label_cnt))
    if miss:
        print("Giá trị KHÔNG khớp (theo nhãn):", dict(miss))

if __name__ == "__main__":
    main()


In [ ]:
!python pdf_to_dataset.py


In [ ]:
import json
d=json.load(open('train_data_vi.json',encoding='utf-8'))
text,ann=d[0]
print(text[:350])
print('\n--- NHÃN ---')
for s,e,l in ann['entities']: print(f'  [{l:20s}] {text[s:e]!r}')


## Bước 3 — Ghi & chạy: JSON → .spacy (chia 80/20)

In [ ]:
%%writefile json_to_spacy_vi.py
# -*- coding: utf-8 -*-
"""
Chuyển train_data_vi.json -> train_vi.spacy / dev_vi.spacy (định dạng spaCy v3).
Dùng tokenizer 'xx' (đa ngôn ngữ) để không phải cài pyvi.
Chạy trên máy có cài spacy (hoặc trên Colab):
    pip install -U spacy spacy-transformers scikit-learn
    python json_to_spacy_vi.py
"""
import json
import spacy
from spacy.tokens import DocBin
from sklearn.model_selection import train_test_split

data = json.load(open("train_data_vi.json", encoding="utf-8"))
train, dev = train_test_split(data, test_size=0.2, random_state=42)
print(f"train={len(train)}  dev={len(dev)}")

def build(rows, path):
    nlp = spacy.blank("xx")           # tokenizer đa ngôn ngữ
    db = DocBin()
    dropped = 0
    for text, annot in rows:
        doc = nlp.make_doc(text)
        ents, occupied = [], set()
        for start, end, label in annot["entities"]:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is None:
                dropped += 1
                continue
            if any(i in occupied for i in range(span.start, span.end)):
                continue
            occupied.update(range(span.start, span.end))
            ents.append(span)
        doc.ents = ents
        db.add(doc)
    db.to_disk(path)
    print(f"  -> {path}  (bỏ {dropped} span lệch token)")

build(train, "train_vi.spacy")
build(dev, "dev_vi.spacy")
print("Xong.")


In [ ]:
!python json_to_spacy_vi.py


## Bước 4 — Cấu hình (transformer `xlm-roberta-base`, LR warmup 5e-5)

In [ ]:
%%writefile base_config_vi.cfg
[paths]
train = null
dev = null

[system]
gpu_allocator = "pytorch"
seed = 42

[nlp]
lang = "xx"
pipeline = ["transformer","ner"]
batch_size = 128

[components]

[components.transformer]
factory = "transformer"

[components.transformer.model]
@architectures = "spacy-transformers.TransformerModel.v3"
name = "xlm-roberta-base"

[components.transformer.model.get_spans]
@span_getters = "spacy-transformers.strided_spans.v1"
window = 128
stride = 96

[components.transformer.model.tokenizer_config]
use_fast = true

[components.ner]
factory = "ner"

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 64
maxout_pieces = 2
use_upper = false
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy-transformers.TransformerListener.v1"
grad_factor = 1.0

[components.ner.model.tok2vec.pooling]
@layers = "reduce_mean.v1"

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0

[training]
accumulate_gradient = 3
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
dropout = 0.1
patience = 400
max_epochs = 0
max_steps = 2000
eval_frequency = 100

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = 0.01
grad_clip = 1.0
use_averages = false
eps = 0.00000001

[training.optimizer.learn_rate]
@schedules = "warmup_linear.v1"
warmup_steps = 250
total_steps = 2000
initial_rate = 0.00005

[training.batcher]
@batchers = "spacy.batch_by_padded.v1"
discard_oversize = true
size = 2000
buffer = 256

[initialize]


In [ ]:
!python -m spacy init fill-config base_config_vi.cfg config_vi.cfg


## Bước 4b — Đo hiệu năng TRƯỚC huấn luyện (baseline)
Dựng pipeline y hệt cấu hình sẽ train nhưng chưa học, chấm trên `dev_vi.spacy` để có mốc so sánh. Ghi ra `metrics_before.json`.

In [ ]:
# Bước 4b — ĐÁNH GIÁ TRƯỚC HUẤN LUYỆN (baseline: model khởi tạo, chưa học)
# Chạy TRƯỚC bước train. Nếu đã train rồi vẫn chạy được: cell chỉ dựng model mới từ config,
# KHÔNG đụng tới output_vi đã có.
import json
from collections import Counter
import spacy
from spacy.util import load_config, load_model_from_config
from spacy.training import Corpus

spacy.prefer_gpu()

def evaluate_ner(nlp, path='./dev_vi.spacy'):
    """So khớp span (start,end,label) giữa dự đoán và nhãn vàng -> TP/FP/FN từng nhãn."""
    tp, fp, fn = Counter(), Counter(), Counter()
    n_pred = n_gold = n_doc = 0
    conf = Counter()          # nhầm nhãn: cùng vị trí, khác nhãn
    for eg in Corpus(path)(nlp):
        gold = {(e.start_char, e.end_char, e.label_) for e in eg.reference.ents}
        pred = {(e.start_char, e.end_char, e.label_) for e in nlp(eg.reference.text).ents}
        n_doc += 1; n_gold += len(gold); n_pred += len(pred)
        for s in pred & gold: tp[s[2]] += 1
        for s in pred - gold: fp[s[2]] += 1
        for s in gold - pred: fn[s[2]] += 1
        gpos = {(s, e): l for s, e, l in gold}
        for s, e, l in pred - gold:
            if (s, e) in gpos: conf[(gpos[(s, e)], l)] += 1
    labels = sorted(set(tp) | set(fp) | set(fn))
    per = {}
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = {'p': p, 'r': r, 'f': 2 * p * r / (p + r) if p + r else 0.0,
                  'tp': tp[l], 'fp': fp[l], 'fn': fn[l]}
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    P = TP / (TP + FP) if TP + FP else 0.0
    R = TP / (TP + FN) if TP + FN else 0.0
    return {'micro': {'p': P, 'r': R, 'f': 2 * P * R / (P + R) if P + R else 0.0,
                      'tp': TP, 'fp': FP, 'fn': FN},
            'per_label': per, 'n_doc': n_doc, 'n_gold': n_gold, 'n_pred': n_pred,
            'confusions': {f'{a}->{b}': c for (a, b), c in conf.items()}}

# Dựng pipeline y hệt cấu hình train nhưng CHƯA huấn luyện (trọng số NER khởi tạo ngẫu nhiên)
cfg  = load_config('config_vi.cfg', overrides={'paths.train': './train_vi.spacy',
                                               'paths.dev':   './dev_vi.spacy'})
nlp0 = load_model_from_config(cfg, auto_fill=True)
nlp0.initialize(lambda: Corpus('./train_vi.spacy')(nlp0))
print('Pipeline:', nlp0.pipe_names)
print('Nhãn đã đăng ký:', nlp0.get_pipe('ner').labels)

before = evaluate_ner(nlp0)
json.dump(before, open('metrics_before.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
m = before['micro']
print(f"\nTRƯỚC HUẤN LUYỆN  ->  P={m['p']*100:.2f}  R={m['r']*100:.2f}  F={m['f']*100:.2f}"
      f"  (dự đoán {before['n_pred']} thực thể / {before['n_gold']} thực thể vàng)")
print('Đã ghi metrics_before.json')


## Bước 5 — Huấn luyện (GPU) → `output_vi/model-best`
Mặc định tối đa 2000 bước, dừng sớm nếu 400 bước không cải thiện.

In [ ]:
# Log huấn luyện được ghi ra train_log.txt để lưu làm minh chứng (Bước 8)
!python -m spacy train config_vi.cfg --output ./output_vi --paths.train ./train_vi.spacy --paths.dev ./dev_vi.spacy --gpu-id 0 2>&1 | tee train_log.txt


## Bước 6 — Kiểm thử trên MỘT PDF thật (đúng luồng inference)

In [ ]:
# Kiểm thử trên 1 PDF: XEM CV trước -> rồi mới chạy NER
import base64, fitz, spacy
from spacy import displacy
from IPython.display import display, HTML

nlp = spacy.load('output_vi/model-best')
PDF = 'cv_pdfs/cv_0003.pdf'      # đổi sang file khác nếu muốn
pdf = fitz.open(PDF)

# --- 1. Hiện CV (render từng trang thành ảnh) ---
imgs = [f"<img src='data:image/png;base64,"
        f"{base64.b64encode(pg.get_pixmap(dpi=110).tobytes('png')).decode()}' "
        f"style='width:794px;border:1px solid #bbb;margin-bottom:12px'/>" for pg in pdf]
display(HTML(f"<h3 style='font:600 15px sans-serif'>📄 {PDF} — {len(pdf)} trang</h3>"
             "<div style='max-height:900px;overflow:auto'>" + "".join(imgs) + "</div>"))

# --- 2. Trích text rồi chạy NER ---
text = '\n'.join(pg.get_text() for pg in pdf)
doc = nlp(text)
display(HTML("<h3 style='font:600 15px sans-serif;margin-top:18px'>🔎 Kết quả trích xuất</h3>"))
if doc.ents:
    for e in doc.ents: print(f'{e.label_:22s} | {e.text}')
else:
    print('(không nhận ra thực thể nào)')
displacy.render(doc, style='ent', jupyter=True)


## Bước 6b — Phân tích kết quả: trước vs sau huấn luyện (12 biểu đồ)
Cần `metrics_before.json` (Bước 4b) và `train_log.txt` (log của Bước 5). Xuất ảnh vào thư mục `charts/` để đưa vào báo cáo.

In [ ]:
# Bước 6b — PHÂN TÍCH KẾT QUẢ: trước vs sau huấn luyện (12 biểu đồ)
import json, os, re, glob
import numpy as np, matplotlib.pyplot as plt
import spacy
from spacy.training import Corpus
from collections import Counter

os.makedirs('charts', exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': .25, 'axes.axisbelow': True})
C_BEF, C_AFT = '#c0504d', '#2e75b6'

# ---------- 1. Số liệu TRƯỚC (chạy Bước 4b) và SAU huấn luyện ----------
if not os.path.exists('metrics_before.json'):
    raise SystemExit('Thiếu metrics_before.json — chạy cell Bước 4b trước.')
before = json.load(open('metrics_before.json', encoding='utf-8'))

spacy.prefer_gpu()
nlp = spacy.load('output_vi/model-best')
try:                       # evaluate_ner được định nghĩa ở Bước 4b
    evaluate_ner
except NameError:
    raise SystemExit('Chạy cell Bước 4b trước để có hàm evaluate_ner().')
after = evaluate_ner(nlp)
json.dump(after, open('metrics_after.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

labels = sorted(after['per_label'])
short  = [l.replace(' Address', '').replace('Companies worked at', 'Companies')
           .replace('College Name', 'College').replace('Years of Experience', 'YoE')
           .replace('Graduation Year', 'Grad.Year') for l in labels]
get = lambda d, l, k: d['per_label'].get(l, {}).get(k, 0)

# ---------- 2. Đọc log huấn luyện ----------
log = None
for f in ['train_log.txt', 'output_vi/train_log.txt']:
    if os.path.exists(f): log = open(f, encoding='utf-8', errors='ignore').read(); break
rows = []
if log:
    for ln in log.splitlines():
        s = re.sub(r'\x1b\[[\d;]*m', '', ln).strip()
        if not re.fullmatch(r'[\d.\s-]+', s):   # bỏ dòng có chữ (thanh tiến trình tải model...)
            continue
        c = s.split()
        if len(c) == 8 and c[0].isdigit() and c[1].isdigit():
            rows.append([float(x) for x in c])
steps = np.array([r[1] for r in rows]) if rows else np.array([])
lt, ln_, ef, ep, er, sc = (np.array([r[i] for r in rows]) for i in (2, 3, 4, 5, 6, 7)) \
    if rows else ([np.array([])] * 6)
if not rows:
    print('! Không thấy train_log.txt -> bỏ qua 4 biểu đồ đường cong huấn luyện.')

# ---------- 3. Thống kê tập dữ liệu ----------
data = json.load(open('train_data_vi.json', encoding='utf-8'))
ent_per_label = Counter(l for _, a in data for *_, l in a['entities'])
doc_chars     = [len(t) for t, _ in data]
ents_per_doc  = [len(a['entities']) for _, a in data]

def save(fig, name):
    fig.tight_layout(); fig.savefig(f'charts/{name}.png', bbox_inches='tight'); plt.show()

# --- H1: P/R/F tổng thể trước vs sau ---
fig, ax = plt.subplots(figsize=(5.2, 3.4)); x = np.arange(3); w = .36
b = [before['micro'][k] * 100 for k in 'prf']; a = [after['micro'][k] * 100 for k in 'prf']
ax.bar(x - w/2, b, w, label='Trước', color=C_BEF); ax.bar(x + w/2, a, w, label='Sau', color=C_AFT)
for i, (u, v) in enumerate(zip(b, a)):
    ax.text(i - w/2, u + 1, f'{u:.1f}', ha='center', fontsize=8)
    ax.text(i + w/2, v + 1, f'{v:.1f}', ha='center', fontsize=8)
ax.set_xticks(x, ['Precision', 'Recall', 'F1']); ax.set_ylim(0, 108); ax.set_ylabel('%')
ax.set_title('H1. Hiệu năng tổng thể trên tập dev'); ax.legend()
save(fig, 'h01_micro_prf')

# --- H2: F1 theo từng nhãn ---
fig, ax = plt.subplots(figsize=(7, 4.2)); y = np.arange(len(labels))
ax.barh(y - .2, [get(before, l, 'f') * 100 for l in labels], .4, label='Trước', color=C_BEF)
ax.barh(y + .2, [get(after,  l, 'f') * 100 for l in labels], .4, label='Sau',   color=C_AFT)
ax.set_yticks(y, short); ax.set_xlabel('F1 (%)'); ax.set_xlim(0, 105)
ax.set_title('H2. F1 theo từng nhãn'); ax.legend(loc='lower right')
save(fig, 'h02_f1_per_label')

# --- H3: Precision & Recall sau huấn luyện, theo nhãn ---
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.barh(y - .2, [get(after, l, 'p') * 100 for l in labels], .4, label='Precision', color='#548235')
ax.barh(y + .2, [get(after, l, 'r') * 100 for l in labels], .4, label='Recall',    color='#bf8f00')
ax.set_yticks(y, short); ax.set_xlim(0, 105); ax.set_xlabel('%')
ax.set_title('H3. Precision / Recall từng nhãn (sau huấn luyện)'); ax.legend(loc='lower right')
save(fig, 'h03_pr_per_label')

# --- H4: TP / FP / FN theo nhãn (sau) ---
fig, ax = plt.subplots(figsize=(7, 4))
tp = np.array([get(after, l, 'tp') for l in labels]); fp = np.array([get(after, l, 'fp') for l in labels])
fn = np.array([get(after, l, 'fn') for l in labels])
ax.bar(short, tp, label='TP (đúng)', color='#548235')
ax.bar(short, fp, bottom=tp, label='FP (thừa)', color='#c0504d')
ax.bar(short, fn, bottom=tp + fp, label='FN (sót)', color='#7f7f7f')
ax.set_ylabel('Số thực thể'); ax.set_title('H4. TP / FP / FN từng nhãn (sau huấn luyện)')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right'); ax.legend()
save(fig, 'h04_tp_fp_fn')

# --- H5: Số thực thể dự đoán: trước vs sau vs nhãn vàng ---
fig, ax = plt.subplots(figsize=(7, 4)); x = np.arange(len(labels)); w = .27
gold_n = [get(after, l, 'tp') + get(after, l, 'fn') for l in labels]
ax.bar(x - w, [get(before, l, 'tp') + get(before, l, 'fp') for l in labels], w, label='Trước', color=C_BEF)
ax.bar(x,     [get(after,  l, 'tp') + get(after,  l, 'fp') for l in labels], w, label='Sau',   color=C_AFT)
ax.bar(x + w, gold_n, w, label='Nhãn vàng', color='#a6a6a6')
ax.set_xticks(x, short); plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
ax.set_ylabel('Số thực thể'); ax.set_title('H5. Số thực thể nhận ra so với nhãn vàng'); ax.legend()
save(fig, 'h05_counts_vs_gold')

# --- H6: Đường cong F1 trên dev theo bước huấn luyện ---
if rows:
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(steps, ef, color=C_AFT, lw=1.8, label='ENTS_F (dev)')
    i = int(np.argmax(ef)); ax.scatter([steps[i]], [ef[i]], color='#c00000', zorder=5)
    ax.annotate(f'best {ef[i]:.2f} @ {int(steps[i])}', (steps[i], ef[i]),
                textcoords='offset points', xytext=(-20, -16), fontsize=8, color='#c00000')
    ax.axhline(before['micro']['f'] * 100, ls='--', color=C_BEF, lw=1.2, label='Trước huấn luyện')
    ax.set_xlabel('Bước'); ax.set_ylabel('F1 (%)'); ax.set_title('H6. F1 trên tập dev theo bước'); ax.legend()
    save(fig, 'h06_f1_curve')

    # --- H7: Loss ---
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(steps, lt, label='LOSS TRANS', color='#7030a0')
    ax.plot(steps, ln_, label='LOSS NER', color='#ed7d31')
    ax.set_yscale('log'); ax.set_xlabel('Bước'); ax.set_ylabel('Loss (log)')
    ax.set_title('H7. Suy giảm hàm mất mát'); ax.legend()
    save(fig, 'h07_loss')

    # --- H8: Precision vs Recall theo bước ---
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(steps, ep, label='Precision', color='#548235')
    ax.plot(steps, er, label='Recall', color='#bf8f00')
    ax.set_xlabel('Bước'); ax.set_ylabel('%'); ax.set_title('H8. Precision / Recall theo bước'); ax.legend()
    save(fig, 'h08_pr_curve')

    # --- H9: Số bước cần để vượt các ngưỡng F1 ---
    fig, ax = plt.subplots(figsize=(5.6, 3.4))
    th = [10, 30, 50, 70, 90, 95, 99]
    need = [int(steps[np.argmax(ef >= t)]) if (ef >= t).any() else np.nan for t in th]
    ax.bar([str(t) for t in th], need, color='#2e75b6')
    for i, v in enumerate(need):
        if not np.isnan(v): ax.text(i, v, str(int(v)), ha='center', va='bottom', fontsize=8)
    ax.set_xlabel('Ngưỡng F1 (%)'); ax.set_ylabel('Bước đầu tiên đạt được')
    ax.set_title('H9. Tốc độ hội tụ'); save(fig, 'h09_convergence')

# --- H10: Phân bố nhãn trong tập dữ liệu ---
fig, ax = plt.subplots(figsize=(7, 3.8))
ks = sorted(ent_per_label, key=ent_per_label.get, reverse=True)
ax.bar([k[:14] for k in ks], [ent_per_label[k] for k in ks], color='#4472c4')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right'); ax.set_ylabel('Số thực thể')
ax.set_title(f'H10. Phân bố nhãn trên {len(data)} CV'); save(fig, 'h10_label_dist')

# --- H11: Phân bố độ dài CV & số thực thể mỗi CV ---
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
axes[0].hist(doc_chars, bins=30, color='#4472c4'); axes[0].set_xlabel('Số ký tự / CV')
axes[0].set_ylabel('Số CV'); axes[0].set_title('Độ dài văn bản')
axes[1].hist(ents_per_doc, bins=range(min(ents_per_doc), max(ents_per_doc) + 2), color='#70ad47')
axes[1].set_xlabel('Số thực thể / CV'); axes[1].set_title('Mật độ nhãn')
fig.suptitle('H11. Đặc điểm tập dữ liệu', y=1.02); save(fig, 'h11_dataset')

# --- H12: Các trường hợp nhầm nhãn (cùng vị trí, sai loại) ---
fig, ax = plt.subplots(figsize=(6.4, 3.4))
cf = after['confusions']
if cf:
    ks = sorted(cf, key=cf.get, reverse=True)[:10]
    ax.barh(ks[::-1], [cf[k] for k in ks][::-1], color='#c0504d')
    ax.set_xlabel('Số lần'); ax.set_title('H12. Nhầm nhãn sau huấn luyện (vàng -> dự đoán)')
else:
    ax.text(.5, .5, 'Không có trường hợp nhầm nhãn\n(mọi span khớp vị trí đều đúng loại)',
            ha='center', va='center', fontsize=11); ax.set_axis_off()
    ax.set_title('H12. Nhầm nhãn sau huấn luyện')
save(fig, 'h12_confusions')

# ---------- 4. Bảng tổng hợp ----------
print(f"\n{'Nhãn':24s}{'F1 trước':>10s}{'F1 sau':>9s}{'P sau':>8s}{'R sau':>8s}{'TP':>5s}{'FP':>5s}{'FN':>5s}")
for l in labels:
    print(f'{l:24s}{get(before,l,"f")*100:>10.2f}{get(after,l,"f")*100:>9.2f}'
          f'{get(after,l,"p")*100:>8.2f}{get(after,l,"r")*100:>8.2f}'
          f'{get(after,l,"tp"):>5d}{get(after,l,"fp"):>5d}{get(after,l,"fn"):>5d}')
mb, ma = before['micro'], after['micro']
print(f'{"TỔNG (micro)":24s}{mb["f"]*100:>10.2f}{ma["f"]*100:>9.2f}{ma["p"]*100:>8.2f}{ma["r"]*100:>8.2f}'
      f'{ma["tp"]:>5d}{ma["fp"]:>5d}{ma["fn"]:>5d}')
print(f'\nĐã lưu {len(glob.glob("charts/*.png"))} biểu đồ trong thư mục charts/')


## Bước 7 (tuỳ chọn) — Upload CV PDF thật của bạn

In [ ]:
# Upload CV PDF thật -> XEM PDF trước -> rồi mới chạy NER
import base64, fitz, spacy
from spacy import displacy
from google.colab import files
from IPython.display import display, HTML

try: nlp
except NameError: nlp = spacy.load('output_vi/model-best')

up = files.upload(); fn = list(up.keys())[0]
pdf = fitz.open(fn)

# --- 1. Hiện PDF vừa upload (render từng trang thành ảnh) ---
imgs = [f"<img src='data:image/png;base64,"
        f"{base64.b64encode(pg.get_pixmap(dpi=110).tobytes('png')).decode()}' "
        f"style='width:794px;border:1px solid #bbb;margin-bottom:12px'/>" for pg in pdf]
display(HTML(f"<h3 style='font:600 15px sans-serif'>📄 {fn} — {len(pdf)} trang</h3>"
             "<div style='max-height:900px;overflow:auto'>" + "".join(imgs) + "</div>"))

# --- 2. Trích text rồi chạy NER ---
text = '\n'.join(pg.get_text() for pg in pdf)
doc = nlp(text)
display(HTML("<h3 style='font:600 15px sans-serif;margin-top:18px'>🔎 Kết quả trích xuất</h3>"))
if doc.ents:
    for e in doc.ents: print(f'{e.label_:22s} | {e.text}')
else:
    print('(không nhận ra thực thể nào)')
displacy.render(doc, style='ent', jupyter=True)


## Bước 8 (tuỳ chọn) — Lưu vào Google Drive

In [ ]:
# Bước 8 — Đóng gói MINH CHỨNG huấn luyện vào Google Drive
import os, json, glob, shutil, hashlib, datetime, subprocess
from google.colab import drive

drive.mount('/content/drive')
STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M')
DST   = f'/content/drive/MyDrive/ResumeParserVI/run_{STAMP}'
os.makedirs(DST, exist_ok=True)
print('Thư mục đích:', DST)

# ---- 1. Model đã train + cấu hình + mã nguồn + dữ liệu ----
ITEMS = ['output_vi', 'config_vi.cfg', 'base_config_vi.cfg', 'gen_cv_pdfs.py',
         'pdf_to_dataset.py', 'json_to_spacy_vi.py', 'train_data_vi.json',
         'train_vi.spacy', 'dev_vi.spacy', 'train_log.txt']
for it in ITEMS:
    if not os.path.exists(it):
        print('  (bỏ qua, không có)', it); continue
    dst = os.path.join(DST, os.path.basename(it))
    if os.path.isdir(it): shutil.copytree(it, dst, dirs_exist_ok=True)
    else:                 shutil.copy2(it, dst)
    print('  đã chép', it)

# ---- 2. Nén bộ CV nguồn (300 PDF + ground-truth) ----
if os.path.isdir('cv_pdfs'):
    shutil.make_archive(os.path.join(DST, 'cv_pdfs'), 'zip', 'cv_pdfs')
    print('  đã nén cv_pdfs.zip')

# ---- 3. Chấm lại model trên tập dev -> metrics_dev.json (P/R/F từng nhãn) ----
ev = subprocess.run(['python', '-m', 'spacy', 'evaluate', 'output_vi/model-best',
                     'dev_vi.spacy', '--output', os.path.join(DST, 'metrics_dev.json')],
                    capture_output=True, text=True)
open(os.path.join(DST, 'evaluate_stdout.txt'), 'w', encoding='utf-8').write(ev.stdout + ev.stderr)
print(ev.stdout[-1500:] or ev.stderr[-1500:])

# ---- 4. Môi trường chạy (GPU, phiên bản thư viện) ----
env = []
for cmd in (['nvidia-smi'], ['python', '-m', 'spacy', 'info'], ['pip', 'freeze']):
    r = subprocess.run(cmd, capture_output=True, text=True)
    env.append('$ ' + ' '.join(cmd) + '\n' + (r.stdout or r.stderr))
open(os.path.join(DST, 'environment.txt'), 'w', encoding='utf-8').write('\n\n'.join(env))

# ---- 5. MANIFEST: kích thước, thời điểm sửa, SHA-256 của từng file model ----
def sha256(f):
    h = hashlib.sha256()
    with open(f, 'rb') as fh:
        for blk in iter(lambda: fh.read(1 << 20), b''): h.update(blk)
    return h.hexdigest()

man = {'created_at': datetime.datetime.now().isoformat(timespec='seconds'), 'files': []}
for f in sorted(glob.glob(os.path.join(DST, '**', '*'), recursive=True)):
    if os.path.isfile(f):
        man['files'].append({
            'path': os.path.relpath(f, DST),
            'bytes': os.path.getsize(f),
            'mtime': datetime.datetime.fromtimestamp(os.path.getmtime(f)).isoformat(timespec='seconds'),
            'sha256': sha256(f) if os.path.getsize(f) < 600 * 1024 * 1024 else 'skipped(too large)'})
meta_p = 'output_vi/model-best/meta.json'
if os.path.exists(meta_p):
    m = json.load(open(meta_p, encoding='utf-8'))
    man['model'] = {k: m.get(k) for k in ('lang', 'spacy_version', 'pipeline', 'labels', 'performance')}
json.dump(man, open(os.path.join(DST, 'MANIFEST.json'), 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)

print(f'\nXong — {len(man["files"])} file trong {DST}')
if 'model' in man:
    print('Điểm NER (meta.json):', json.dumps(man['model'].get('performance', {}), ensure_ascii=False)[:400])


## Bước 9 — Chạy bằng weight đã lưu trên Drive
Nạp thẳng `model-best` từ Drive, không cần huấn luyện lại.

In [ ]:
# CELL A — Chạy bằng WEIGHT ĐÃ LƯU TRÊN DRIVE (không huấn luyện lại)
import glob, os, base64, spacy, fitz
from google.colab import drive
from IPython.display import display, HTML
from spacy import displacy

drive.mount('/content/drive')

# tự dò model-best mới nhất trong MyDrive/ResumeParserVI (mọi thư mục con)
cands = sorted(glob.glob('/content/drive/MyDrive/ResumeParserVI/**/model-best', recursive=True),
               key=os.path.getmtime)
assert cands, 'Không thấy model-best trong MyDrive/ResumeParserVI — kiểm tra lại đường dẫn Drive.'
MODEL = cands[-1]
print('Nạp model:', MODEL)

spacy.prefer_gpu()
nlp_drive = spacy.load(MODEL)          # ~30-60s vì đọc ~1GB từ Drive
print('Nhãn:', nlp_drive.get_pipe('ner').labels)

def parse_pdf(path, show_pdf=True, nlp=None):
    """Hiện CV rồi trích thông tin bằng model nạp từ Drive."""
    nlp = nlp or nlp_drive
    pdf = fitz.open(path)
    if show_pdf:
        imgs = [f"<img src='data:image/png;base64,"
                f"{base64.b64encode(pg.get_pixmap(dpi=110).tobytes('png')).decode()}' "
                f"style='width:794px;border:1px solid #bbb;margin-bottom:12px'/>" for pg in pdf]
        display(HTML(f"<h3 style='font:600 15px sans-serif'>📄 {os.path.basename(path)}</h3>"
                     "<div style='max-height:900px;overflow:auto'>" + ''.join(imgs) + '</div>'))
    doc = nlp('\n'.join(pg.get_text() for pg in pdf))
    display(HTML("<h3 style='font:600 15px sans-serif'>🔎 Kết quả trích xuất</h3>"))
    for e in doc.ents:
        print(f'{e.label_:22s} | {" ".join(e.text.split())}')
    if not doc.ents:
        print('(không nhận ra thực thể nào)')
    displacy.render(doc, style='ent', jupyter=True)
    return doc

src = sorted(glob.glob('cv_pdfs/*.pdf'))
if src:
    parse_pdf(src[2])                  # đổi chỉ số để xem CV khác
else:
    from google.colab import files
    up = files.upload(); parse_pdf(list(up.keys())[0])


## Bước 10 — Runtime mới hoàn toàn: dùng model trên Drive, không huấn luyện
Chạy được trên session trống: cài thư viện, kéo model + dữ liệu từ Drive về rồi trích xuất.

In [ ]:
# CELL B — SESSION TRỐNG / RUNTIME MỚI: dùng model từ Drive, KHÔNG huấn luyện gì cả
# (chạy được cả khi /content rỗng: không cần cv_pdfs, không cần train_vi.spacy)
!pip -q install spacy spacy-transformers pymupdf

import glob, os, shutil, json, spacy, fitz
from google.colab import drive, files
from IPython.display import display, HTML
from spacy import displacy

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/ResumeParserVI'

# 1) Lấy model-best mới nhất; chép về ổ cục bộ cho nhanh (đọc thẳng từ Drive rất chậm)
cands = sorted(glob.glob(f'{ROOT}/**/model-best', recursive=True), key=os.path.getmtime)
assert cands, f'Không thấy model-best trong {ROOT}'
if not os.path.isdir('model-best'):
    print('Đang chép model từ Drive về /content (~1GB, 1-2 phút)...')
    shutil.copytree(cands[-1], 'model-best')
spacy.prefer_gpu()
nlp = spacy.load('model-best')
print('Model:', cands[-1])
print('Nhãn :', nlp.get_pipe('ner').labels)

# 2) (tuỳ chọn) Chấm lại trên tập dev đã lưu -> chứng minh model hoạt động, không cần train
dev = glob.glob(f'{ROOT}/**/dev_vi.spacy', recursive=True)
if dev:
    shutil.copy(dev[-1], 'dev_vi.spacy')
    from spacy.training import Corpus
    from collections import Counter
    tp = fp = fn = 0
    for eg in Corpus('./dev_vi.spacy')(nlp):
        gold = {(e.start_char, e.end_char, e.label_) for e in eg.reference.ents}
        pred = {(e.start_char, e.end_char, e.label_) for e in nlp(eg.reference.text).ents}
        tp += len(pred & gold); fp += len(pred - gold); fn += len(gold - pred)
    P, R = tp / (tp + fp), tp / (tp + fn)
    print(f'\nĐánh giá lại trên dev: P={P*100:.2f}  R={R*100:.2f}  F={2*P*R/(P+R)*100:.2f}'
          f'  (TP={tp} FP={fp} FN={fn})')
else:
    print('(không thấy dev_vi.spacy trên Drive -> bỏ qua bước chấm điểm)')

# 3) Trích thông tin từ một CV: lấy trong cv_pdfs.zip trên Drive, hoặc upload file của bạn
zips = glob.glob(f'{ROOT}/**/cv_pdfs.zip', recursive=True)
if zips and not os.path.isdir('cv_pdfs'):
    shutil.unpack_archive(zips[-1], 'cv_pdfs')
src = sorted(glob.glob('cv_pdfs/*.pdf'))
path = src[2] if src else list(files.upload().keys())[0]

pdf = fitz.open(path)
import base64
imgs = [f"<img src='data:image/png;base64,"
        f"{base64.b64encode(pg.get_pixmap(dpi=110).tobytes('png')).decode()}' "
        f"style='width:794px;border:1px solid #bbb;margin-bottom:12px'/>" for pg in pdf]
display(HTML(f"<h3 style='font:600 15px sans-serif'>📄 {os.path.basename(path)}</h3>"
             "<div style='max-height:900px;overflow:auto'>" + ''.join(imgs) + '</div>'))
doc = nlp('\n'.join(pg.get_text() for pg in pdf))
display(HTML("<h3 style='font:600 15px sans-serif'>🔎 Kết quả trích xuất</h3>"))
for e in doc.ents:
    print(f'{e.label_:22s} | {" ".join(e.text.split())}')
displacy.render(doc, style='ent', jupyter=True)
